# Bài 4
Đây là notebook chứa mã nguồn đầy đủ của bài 4.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai04(budget=50000, w_gdp=0.40, w_equity=0.25, w_ai=0.20, fairness_cv=0.30):
    regions = ['NMM', 'RRD', 'NCC', 'CH', 'SE', 'MD']
    items   = ['I', 'D', 'AI', 'H']
    
    beta = {
        ('NMM','I'):1.15, ('NMM','D'):0.85, ('NMM','AI'):0.55, ('NMM','H'):1.30,
        ('RRD','I'):0.95, ('RRD','D'):1.25, ('RRD','AI'):1.40, ('RRD','H'):1.05,
        ('NCC','I'):1.05, ('NCC','D'):0.95, ('NCC','AI'):0.85, ('NCC','H'):1.15,
        ('CH','I') :1.20, ('CH','D') :0.75, ('CH','AI') :0.45, ('CH','H') :1.35,
        ('SE','I') :0.90, ('SE','D') :1.30, ('SE','AI') :1.55, ('SE','H') :1.00,
        ('MD','I') :1.10, ('MD','D') :0.85, ('MD','AI') :0.65, ('MD','H') :1.25
    }
    
    beta_adj = {}
    for r in regions:
        for j in items:
            val = beta[(r, j)]
            if j == 'AI': val *= (1.0 + w_ai)
            elif j == 'H': val *= (1.0 + w_equity)
            beta_adj[(r, j)] = val * (1.0 + w_gdp)
            
    D0 = {'NMM':38, 'RRD':78, 'NCC':55, 'CH':32, 'SE':82, 'MD':48}
    gamma_val = 0.002
    lam = 0.6
    
    def solve_pulp(use_c3=True, use_c5=True):
        m = pulp.LpProblem('VN_Digital_Budget', pulp.LpMaximize)
        x = pulp.LpVariable.dicts('x', (regions, items), lowBound=0)
        m += pulp.lpSum(beta_adj[(r, j)] * x[r][j] for r in regions for j in items)
        m += pulp.lpSum(x[r][j] for r in regions for j in items) <= budget
        if use_c3:
            for r in regions:
                m += pulp.lpSum(x[r][j] for j in items) >= 5000
                m += pulp.lpSum(x[r][j] for j in items) <= 12000
        m += pulp.lpSum(x[r]['H'] for r in regions) >= 12000
        if use_c5:
            Dmax = pulp.LpVariable('Dmax')
            for r in regions:
                m += D0[r] + gamma_val * x[r]['D'] <= Dmax
                m += D0[r] + gamma_val * x[r]['D'] >= lam * Dmax
        m.solve(pulp.PULP_CBC_CMD(msg=False))
        ok = m.status == pulp.LpStatusOptimal
        if ok:
            alloc_table = {r: {j: round(pulp.value(x[r][j]), 2) for j in items} for r in regions}
            z = pulp.value(m.objective)
            return True, z, alloc_table
        return False, 0, {}

    # 1. PuLP Base
    ok, z, alloc_table = solve_pulp(use_c3=True, use_c5=True)
    
    # 2. CVXPY Base
    cvxpy_ok, cvxpy_z, cvxpy_alloc = False, 0.0, {}
    try:
        import cvxpy as cp
        X = cp.Variable((len(regions), len(items)), nonneg=True)
        B_mat = np.array([[beta_adj[(r, j)] for j in items] for r in regions])
        objective = cp.Maximize(cp.sum(cp.multiply(B_mat, X)))
        constraints = [
            cp.sum(X) <= budget,
            cp.sum(X, axis=1) >= 5000,
            cp.sum(X, axis=1) <= 12000,
            cp.sum(X[:, 3]) >= 12000
        ]
        D_idx = 1
        D0_arr = np.array([D0[r] for r in regions])
        D_score = D0_arr + gamma_val * X[:, D_idx]
        Dmax = cp.Variable()
        constraints += [
            D_score <= Dmax,
            D_score >= lam * Dmax
        ]
        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.SCS)
        if prob.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
            cvxpy_ok = True
            cvxpy_z = float(prob.value)
            cvxpy_alloc = {regions[i]: {items[j]: round(float(X.value[i,j]), 2) for j in range(len(items))} for i in range(len(regions))}
    except ImportError:
        pass
        
    # 3. PuLP No Equity C5
    ok_noeq, z_noeq, alloc_noeq = solve_pulp(use_c3=True, use_c5=False)
    
    # 4. PuLP No Region Bounds C3
    ok_noc3, z_noc3, alloc_noc3 = solve_pulp(use_c3=False, use_c5=True)
    
    return {
        'status': 'Optimal' if ok else 'Infeasible',
        'Z': z,
        'allocation': alloc_table,
        'cvxpy_ok': cvxpy_ok,
        'cvxpy_z': cvxpy_z,
        'cvxpy_alloc': cvxpy_alloc,
        'no_equity_z': z_noeq,
        'no_equity_alloc': alloc_noeq,
        'equity_cost': z_noeq - z if ok and ok_noeq else 0,
        'noc3_z': z_noc3,
        'noc3_alloc': alloc_noc3,
        'c3_cost': z_noc3 - z if ok and ok_noc3 else 0,
        'feasible': ok,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai04()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())